In [0]:
CREATE OR REPLACE TEMP VIEW runtime_parameters AS

SELECT
    (SELECT MAX(service_date) FROM com_edp_prd.com_raw.kom_medical_events) AS max_medical_date,

    (SELECT MAX(fill_date) FROM com_edp_prd.com_raw.kom_pharmacy_events) AS max_pharmacy_date,

    LAST_DAY(
        ADD_MONTHS(
            LEAST(
                (SELECT MAX(service_date) FROM com_edp_prd.com_raw.kom_medical_events),
                (SELECT MAX(fill_date) FROM com_edp_prd.com_raw.kom_pharmacy_events)
            ), -1
        )
    ) AS end_date,

    CURRENT_DATE() AS run_date;

    SELECT * FROM runtime_parameters;

In [0]:
CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.claims_patients_avlayah AS

SELECT
    c.patient_id,
    c.service_date,
    master.latest_insurance_type,
    master.patient_age,
    master.severity,
    pt.territory_id,
    pt.territory_name,
    tr.region_id,
    tr.region_name

FROM (

    
    SELECT DISTINCT
        PATIENT_ID AS patient_id,
        SERVICE_DATE AS service_date
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE NDC11 = '8497600101' 
    
    UNION ALL

    SELECT DISTINCT
        PATIENT_ID AS patient_id,
        FILL_DATE AS service_date
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE NDC11 = '8497600101'
    and TRANSACTION_RESULT = 'PAID'

    UNION ALL

    SELECT DISTINCT
        PATIENT_ID AS patient_id,
        SERVICE_DATE AS service_date
    FROM com_edp_prd.com_raw.kom_medical_events
    where PROCEDURE_CODE  IN ('J3490', 'J3590', 'J9999')
    and SERVICE_DATE >= DATE('2026-03-01')
) c

LEFT JOIN (
    
    SELECT DISTINCT
        patient_id ,
        primary_hcp_territory_id_2yr AS territory_id,
        primary_hcp_territory_2yr AS territory_name
    FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master
    WHERE primary_hcp_territory_id_2yr IS NOT NULL
) pt
    ON c.patient_id = pt.patient_id

LEFT JOIN (
    
    SELECT
        territory_id,
        MAX(region_id) AS region_id,
        MAX(region_name) AS region_name
    FROM com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping
    GROUP BY territory_id
) tr
    ON pt.territory_id = tr.territory_id
left join com_edp_prd.cmpa_insights_internal_schema.patient360_master master
    on c.patient_id = master.patient_id


In [0]:
select * from com_edp_prd.cmpa_insights_internal_schema.claims_patients_avlayah

In [0]:
CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.sp_patients_avlayah AS (
  SELECT DISTINCT
    d.crx_patient_id AS patient_id,
    d.ship_date AS service_date,
    z.region_id,
    z.region_name,
    z.territory_id,
    z.territory_name
  FROM com_edp_prd.com_intgr.sp_dispense d
  LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping z
    ON REGEXP_EXTRACT(d.shipment_hcp_zip, '^[0-9]{5}', 0) = CAST(z.zipcode AS STRING)
  WHERE d.ndc IN ('8497600101')
    AND d.fill_type = 'Paid'
    AND d.returned_flag = 'N'
)

In [0]:
CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.hub_patients_avlayah AS (
 
  WITH occam_status AS (
    SELECT DISTINCT
        s.crx_patient_id AS patient_id,
        CAST(s.status_datetime AS DATE) AS service_date
    FROM com_edp_prd.com_intgr.sp_status s
    WHERE UPPER(TRIM(s.status_source)) ilike '%OCCAM%'
      AND s.crx_patient_id IS NOT NULL
      AND s.status_datetime IS NOT NULL
  )
 
  SELECT DISTINCT
      o.patient_id,
      o.service_date,
      z.region_id,
      z.region_name,
      z.territory_id,
      z.territory_name
 
  FROM occam_status o
 
  LEFT JOIN com_edp_prd.com_intgr.sp_patients p
      ON o.patient_id = p.crx_patient_id
 
  LEFT JOIN com_edp_prd.com_intgr.sp_hcp h
      ON p.current_crx_hcp_id = h.crx_hcp_id
 
  LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping z
      ON REGEXP_EXTRACT(
            COALESCE(h.hcp_address_zip_postal_code, h.`hcp_address_zip_postal_code`),
            '^[0-9]{5}', 0
         ) = CAST(z.zipcode AS STRING)
 
);

In [0]:
CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.crm_patients_avlayah AS (

  SELECT
      z.region_id,
      z.region_name,
      z.territory_id,
      z.territory_name,
      DATE(a.modified_date__v) AS service_date,   
      SUM(COALESCE(a.dnli_tivi_patients__c,0)) AS crm_patient_count

  FROM com_edp_prd.com_raw.vcrm_account__v a

  LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping z
      ON SUBSTRING(a.postal_code_cda__v, 1, 5) = CAST(z.zipcode AS STRING)

  WHERE a.ispersonaccount__v = false   
    AND a.npi__v IS NOT NULL
    AND a.dnli_tivi_patients__c IS NOT NULL

  GROUP BY
      z.region_id,
      z.region_name,
      z.territory_id,
      z.territory_name,
      DATE(a.modified_date__v)
)

In [0]:
select * from com_edp_prd.cmpa_insights_internal_schema.crm_patients_avlayah

In [0]:
CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.hcp_prescribed_avlayah AS

WITH base_patients AS (
    SELECT DISTINCT patient_id
    FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master
),

all_hcp_events AS (

    -- MEDICAL NDC
    SELECT DISTINCT
        p.patient_id,
        m.service_date,
        TRIM(COALESCE(m.rendering_npi, m.referring_npi)) AS npi
    FROM base_patients p
    INNER JOIN com_edp_prd.com_raw.kom_medical_events m
        ON p.patient_id = m.patient_id
    WHERE TRIM(m.ndc11) = '8497600101'
      AND COALESCE(m.rendering_npi, m.referring_npi) IS NOT NULL
      AND TRIM(COALESCE(m.rendering_npi, m.referring_npi)) <> ''

    UNION ALL

    -- PHARMACY NDC
    SELECT DISTINCT
        p.patient_id,
        r.fill_date AS service_date,
        TRIM(r.prescriber_npi) AS npi
    FROM base_patients p
    INNER JOIN com_edp_prd.com_raw.kom_pharmacy_events r
        ON p.patient_id = r.patient_id
    WHERE TRIM(r.ndc11) = '8497600101'
      AND UPPER(TRIM(r.transaction_result)) = 'PAID'
      AND r.prescriber_npi IS NOT NULL
      AND TRIM(r.prescriber_npi) <> ''

    UNION ALL

    -- PROCEDURE CODE
    SELECT DISTINCT
        p.patient_id,
        m.service_date,
        TRIM(COALESCE(m.rendering_npi, m.referring_npi)) AS npi
    FROM base_patients p
    INNER JOIN com_edp_prd.com_raw.kom_medical_events m
        ON p.patient_id = m.patient_id
    WHERE TRIM(m.procedure_code) IN ('J3490', 'J3590', 'J9999')
      AND COALESCE(m.rendering_npi, m.referring_npi) IS NOT NULL
      AND TRIM(COALESCE(m.rendering_npi, m.referring_npi)) <> ''
      AND m.service_date >= DATE('2026-03-01')
),

hcp_geo AS (
    SELECT DISTINCT
        e.service_date,
        e.npi,
        z.region_id,
        z.region_name,
        z.territory_id,
        z.territory_name
    FROM all_hcp_events e
    LEFT JOIN com_edp_prd.com_raw.kom_providers kp
        ON TRIM(e.npi) = TRIM(kp.npi)
    LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping z
        ON LEFT(TRIM(kp.provider_zip), 5) = LEFT(TRIM(CAST(z.zipcode AS STRING)), 5)
    WHERE e.npi IS NOT NULL
      AND z.territory_id IS NOT NULL
)

SELECT
    service_date,
    npi,
    region_id,
    region_name,
    territory_id,
    territory_name
FROM hcp_geo;

In [0]:
CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.hco_ordered_avlayah AS

WITH accounts_dedup AS (
    SELECT *
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (
                   PARTITION BY crx_account_id
                   ORDER BY ingestion_date DESC
               ) AS rn
        FROM com_edp_prd.com_intgr.distribution_accounts
    )
    WHERE rn = 1
),

-- shipments_dedup AS (
--     SELECT *
--     FROM (
--         SELECT *,
--                ROW_NUMBER() OVER (
--                    PARTITION BY invoice_number
--                    ORDER BY ingestion_date DESC
--                ) AS rn
--         FROM com_edp_prd.com_intgr.distribution_sd_shipments
--         WHERE ndc = '84976-0001-01'  
--         AND is_current = true
--     )
--     WHERE rn = 1
-- ),

shipments_dedup AS (
    SELECT *
    FROM com_edp_prd.com_intgr.distribution_sd_shipments
    WHERE ndc = '84976-0001-01'
      AND is_current = true
),

zip_map AS (
    SELECT DISTINCT
        LEFT(REGEXP_REPLACE(zipcode, '[^0-9]', ''), 5) AS zip5,
        region_id,
        region_name,
        territory_id,
        territory_name
    FROM com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping
),

base AS (
    SELECT *
    FROM (
        SELECT
            CAST(s.order_date AS DATE) AS service_date,
            s.crx_account_id,

            CASE 
                WHEN UPPER(a.account_type) = 'SP'
                     OR UPPER(a.account_facility_name) LIKE '%ORSINI%'
                THEN 'SP'
                ELSE 'HCO'
            END AS channel,

            z.region_id,
            z.region_name,
            z.territory_id,
            z.territory_name,

            ROW_NUMBER() OVER (
                PARTITION BY s.crx_account_id
                ORDER BY s.order_date ASC   
            ) AS rn

        FROM shipments_dedup s
        LEFT JOIN accounts_dedup a
            ON s.crx_account_id = a.crx_account_id
        LEFT JOIN zip_map z
            ON LEFT(REGEXP_REPLACE(a.account_facility_zip, '[^0-9]', ''), 5) = z.zip5

        WHERE z.territory_id IS NOT NULL
    )
    WHERE rn = 1   
)

SELECT
    service_date,
    region_id,
    region_name,
    territory_id,
    territory_name,

    COUNT(DISTINCT crx_account_id) AS hco_accounts_ordered

FROM base
WHERE channel = 'HCO'

GROUP BY
    service_date,
    region_id,
    region_name,
    territory_id,
    territory_name;


In [0]:
select * from com_edp_prd.com_intgr.distribution_sd_shipments

In [0]:
Select * from com_edp_prd.cmpa_insights_internal_schema.hco_ordered_avlayah 

In [0]:
CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.vials_avlayah AS

WITH accounts_dedup AS (
    SELECT *
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (
                   PARTITION BY crx_account_id 
                   ORDER BY ingestion_date DESC
               ) AS rn
        FROM com_edp_prd.com_intgr.distribution_accounts
    )
    WHERE rn = 1
),

shipments_dedup AS (
    SELECT *
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (
                   PARTITION BY invoice_number
                   ORDER BY ingestion_date DESC
               ) AS rn
        FROM com_edp_prd.com_intgr.distribution_sd_shipments
        WHERE ndc = '84976-0001-01'
    )
    WHERE rn = 1
),

zip_map AS (
    SELECT DISTINCT
        LEFT(REGEXP_REPLACE(zipcode, '[^0-9]', ''), 5) AS zip5,
        region_id,
        region_name,
        territory_id,
        territory_name
    FROM com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping
),

base_shipments AS (
    SELECT
        CAST(s.order_date AS DATE) AS service_date,
        s.crx_account_id,
        s.quantity_shipped,
        CASE 
            WHEN UPPER(a.account_type) = 'SP' 
                 OR UPPER(a.account_facility_name) LIKE '%ORSINI%' 
            THEN 'SP'
            ELSE 'HCO'
        END AS final_channel,

        z.region_id,
        z.region_name,
        z.territory_id,
        z.territory_name
    FROM shipments_dedup s
    LEFT JOIN accounts_dedup a
        ON s.crx_account_id = a.crx_account_id
    LEFT JOIN zip_map z
        ON LEFT(REGEXP_REPLACE(a.account_facility_zip, '[^0-9]', ''), 5) = z.zip5
    WHERE z.territory_id IS NOT NULL
),

vials_agg AS (
    SELECT
        service_date,
        region_id,
        region_name,
        territory_id,
        territory_name,

        SUM(quantity_shipped) AS total_vials,

        SUM(CASE 
                WHEN final_channel = 'SP' THEN quantity_shipped 
                ELSE 0 
            END) AS sp_vials,

        SUM(CASE 
                WHEN final_channel = 'HCO' THEN quantity_shipped 
                ELSE 0 
            END) AS hco_vials
    FROM base_shipments
    GROUP BY
        service_date,
        region_id,
        region_name,
        territory_id,
        territory_name
)

SELECT
    service_date,
    region_id,
    region_name,
    territory_id,
    territory_name,
    total_vials,
    sp_vials,
    hco_vials,
    ROUND(CASE WHEN total_vials > 0 THEN sp_vials * 1.0 / total_vials ELSE 0 END, 4) AS sp_vials_pct,
    ROUND(CASE WHEN total_vials > 0 THEN hco_vials * 1.0 / total_vials ELSE 0 END, 4) AS hco_vials_pct
FROM vials_agg;

In [0]:
-- CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.vials_avlayah AS

-- WITH base_dispense AS (
--     SELECT
--         CAST(d.ship_date AS DATE) AS service_date,
--         d.crx_hcp_id,
--         d.quantity,
--         d.pharmacy_name,
--         z.region_id,
--         z.region_name,
--         z.territory_id,
--         z.territory_name
--     FROM com_edp_prd.com_intgr.sp_dispense d
--     LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping z
--         ON LEFT(TRIM(CAST(d.shipment_hcp_zip AS STRING)), 5) = LEFT(TRIM(CAST(z.zipcode AS STRING)), 5)
--     WHERE z.territory_id IS NOT NULL
-- ),

-- hub_hcp AS (
--     SELECT DISTINCT
--         current_crx_hcp_id AS crx_hcp_id
--     FROM com_edp_prd.com_intgr.sp_patients
--     WHERE latest_status_source = 'Occam'
--       AND current_crx_hcp_id IS NOT NULL
-- )

-- SELECT
--     b.service_date,
--     b.region_id,
--     b.region_name,
--     b.territory_id,
--     b.territory_name,
--     b.quantity AS total_vials,
--     CASE
--         WHEN LOWER(b.pharmacy_name) LIKE '%orsini%' THEN b.quantity
--         ELSE 0
--     END AS sp_vials,
--     CASE
--         WHEN h.crx_hcp_id IS NOT NULL THEN b.quantity
--         ELSE 0
--     END AS HCO_VIALS
-- FROM base_dispense b
-- LEFT JOIN hub_hcp h
--     ON b.crx_hcp_id = h.crx_hcp_id;

In [0]:
CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.all_metrics_avlayah AS (

-- CLAIMS
SELECT 
    service_date,
    region_id,
    region_name,
    territory_id,
    territory_name,
    'CLAIMS_PATIENTS' AS metric_name,
    COALESCE(COUNT(DISTINCT patient_id),0) AS metric_value
FROM com_edp_prd.cmpa_insights_internal_schema.claims_patients_avlayah
GROUP BY 1,2,3,4,5,6

UNION ALL

-- SP
SELECT 
    service_date,
    region_id,
    region_name,
    territory_id,
    territory_name,
    'SP_PATIENTS',
    COALESCE(COUNT(DISTINCT patient_id),0)
FROM com_edp_prd.cmpa_insights_internal_schema.sp_patients_avlayah
GROUP BY 1,2,3,4,5,6

UNION ALL

-- HUB
SELECT 
    service_date,
    region_id,
    region_name,
    territory_id,
    territory_name,
    'HUB_PATIENTS',
    COALESCE(COUNT(DISTINCT patient_id),0)
FROM com_edp_prd.cmpa_insights_internal_schema.hub_patients_avlayah
GROUP BY 1,2,3,4,5,6

UNION ALL

-- CRM (NO DISTINCT)
SELECT 
    service_date,
    region_id,
    region_name,
    territory_id,
    territory_name,
    'CRM_PATIENTS',
    COALESCE(SUM(crm_patient_count),0)
FROM com_edp_prd.cmpa_insights_internal_schema.crm_patients_avlayah
GROUP BY 1,2,3,4,5,6

);

In [0]:
CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.overview_aldashboard_metrics_period_insurance AS (

-- LTD
SELECT
    region_id,
    region_name,
    territory_id,
    territory_name,
    CASE 
        WHEN UPPER(latest_insurance_type) LIKE '%MEDICAID%' THEN 'Medicaid'
        WHEN UPPER(latest_insurance_type) LIKE '%MEDICARE%' THEN 'Medicare'
        WHEN UPPER(latest_insurance_type) LIKE '%COMMERCIAL%' THEN 'Commercial'
        ELSE 'Unknown'
    END AS insurance_group,
    'LTD' AS period_type,
    'PATIENT_COUNT' AS metric_name,
    COUNT(DISTINCT patient_id) AS metric_value
FROM com_edp_prd.cmpa_insights_internal_schema.claims_patients_avlayah
WHERE service_date >= DATE('2026-03-01')
GROUP BY 1,2,3,4,5,6,7

UNION ALL

-- YTD
SELECT
    region_id,
    region_name,
    territory_id,
    territory_name,
    CASE 
        WHEN UPPER(latest_insurance_type) LIKE '%MEDICAID%' THEN 'Medicaid'
        WHEN UPPER(latest_insurance_type) LIKE '%MEDICARE%' THEN 'Medicare'
        WHEN UPPER(latest_insurance_type) LIKE '%COMMERCIAL%' THEN 'Commercial'
        ELSE 'Unknown'
    END AS insurance_group,
    'YTD' AS period_type,
    'PATIENT_COUNT' AS metric_name,
    COUNT(DISTINCT patient_id) AS metric_value
FROM com_edp_prd.cmpa_insights_internal_schema.claims_patients_avlayah
WHERE service_date >= DATE_TRUNC('year', (SELECT end_date FROM runtime_parameters))
GROUP BY 1,2,3,4,5,6,7

UNION ALL

-- QTD
SELECT
    region_id,
    region_name,
    territory_id,
    territory_name,
    CASE 
        WHEN UPPER(latest_insurance_type) LIKE '%MEDICAID%' THEN 'Medicaid'
        WHEN UPPER(latest_insurance_type) LIKE '%MEDICARE%' THEN 'Medicare'
        WHEN UPPER(latest_insurance_type) LIKE '%COMMERCIAL%' THEN 'Commercial'
        ELSE 'Unknown'
    END AS insurance_group,
    'QTD' AS period_type,
    'PATIENT_COUNT' AS metric_name,
    COUNT(DISTINCT patient_id) AS metric_value
FROM com_edp_prd.cmpa_insights_internal_schema.claims_patients_avlayah
WHERE service_date >= DATE_TRUNC('quarter', (SELECT end_date FROM runtime_parameters))
GROUP BY 1,2,3,4,5,6,7

UNION ALL

-- MTD
SELECT
    region_id,
    region_name,
    territory_id,
    territory_name,
    CASE 
        WHEN UPPER(latest_insurance_type) LIKE '%MEDICAID%' THEN 'Medicaid'
        WHEN UPPER(latest_insurance_type) LIKE '%MEDICARE%' THEN 'Medicare'
        WHEN UPPER(latest_insurance_type) LIKE '%COMMERCIAL%' THEN 'Commercial'
        ELSE 'Unknown'
    END AS insurance_group,
    'MTD' AS period_type,
    'PATIENT_COUNT' AS metric_name,
    COUNT(DISTINCT patient_id) AS metric_value
FROM com_edp_prd.cmpa_insights_internal_schema.claims_patients_avlayah
WHERE service_date >= DATE_TRUNC('month', (SELECT end_date FROM runtime_parameters))
GROUP BY 1,2,3,4,5,6,7

);

In [0]:
CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.overview_aldashboard_metrics_period AS

WITH runtime AS (
    SELECT
        end_date,
        DATE('2026-03-01')                      AS ltd_start,
        date('2023-08-01')                      As ltd_start_Elaprase,
        DATE_TRUNC('year', end_date)           AS ytd_start,
        DATE_TRUNC('quarter', end_date)        AS qtd_start,
        DATE_TRUNC('month', end_date)          AS mtd_start
    FROM runtime_parameters
),

periods AS (
    SELECT 'LTD' period_type, ltd_start start_date FROM runtime
    UNION ALL
    SELECT 'YTD', ytd_start FROM runtime
    UNION ALL
    SELECT 'QTD', qtd_start FROM runtime
    UNION ALL
    SELECT 'MTD', mtd_start FROM runtime
),

periods_elaprase AS (
    SELECT 'LTD' period_type, ltd_start_elaprase start_date FROM runtime
    UNION ALL
    SELECT 'YTD', ytd_start FROM runtime
    UNION ALL
    SELECT 'QTD', qtd_start FROM runtime
    UNION ALL
    SELECT 'MTD', mtd_start FROM runtime
),

-- ============================================
-- DIMENSIONS
-- ============================================
dim_geo AS (
    SELECT DISTINCT
        region_id,
        region_name,
        territory_id,
        territory_name
    FROM com_edp_prd.cmpa_insights_internal_schema.all_metrics_avlayah
),

dim_metrics AS (
    SELECT DISTINCT metric_name
    FROM (

        SELECT metric_name
        FROM com_edp_prd.cmpa_insights_internal_schema.all_metrics_avlayah

        UNION ALL SELECT 'AVLAYAH_Medicare_Patients'
        UNION ALL SELECT 'AVLAYAH_Medicaid_Patients'
        UNION ALL SELECT 'AVLAYAH_Commercial_Patients'
        UNION ALL SELECT 'AVLAYAH_Unknown_Patients'

        UNION ALL SELECT 'AVLAYAH_PATIENTS_LT_5'
        UNION ALL SELECT 'AVLAYAH_PATIENTS_5_10'
        UNION ALL SELECT 'AVLAYAH_PATIENTS_11_16'
        UNION ALL SELECT 'AVLAYAH_PATIENTS_17_PLUS'

        UNION ALL SELECT 'Severe_Avlayah'
        UNION ALL SELECT 'Attenuated_Avlayah'

        UNION ALL SELECT 'ELAPRASE_LT_5'
        UNION ALL SELECT 'ELAPRASE_5_10'
        UNION ALL SELECT 'ELAPRASE_11_16'
        UNION ALL SELECT 'ELAPRASE_17_PLUS'

        UNION ALL SELECT 'HCP_PRESCRIBED'
        UNION ALL SELECT 'HCO_ORDERED'

        UNION ALL SELECT 'TOTAL_VIALS'
        UNION ALL SELECT 'SP_VIALS'
        UNION ALL SELECT 'HCO_VIALS'

        UNION ALL SELECT 'Avlayah_Claims'
        UNION ALL SELECT 'Avlayah_SP'
        UNION ALL SELECT 'Avlayah_HUB'
        UNION ALL SELECT 'Avlayah_CRM'
    )
),

-- ============================================
-- DEDUP CRM
-- ============================================
crm_dedup AS (
    SELECT
        region_id,
        region_name,
        territory_id,
        territory_name,
        service_date,
        SUM(crm_patient_count) AS crm_patient_count
    FROM com_edp_prd.cmpa_insights_internal_schema.crm_patients_avlayah
    GROUP BY 1,2,3,4,5
),

claims_base AS (
    SELECT *
    FROM com_edp_prd.cmpa_insights_internal_schema.claims_patients_avlayah
),

fact_data AS (

-- ============================================
-- ALL_METRICS_AVLAYAH
-- ============================================
SELECT
    a.region_id,
    a.region_name,
    a.territory_id,
    a.territory_name,
    p.period_type,
    a.metric_name,
    SUM(a.metric_value) AS metric_value
FROM com_edp_prd.cmpa_insights_internal_schema.all_metrics_avlayah a
JOIN periods p
    ON p.start_date IS NOT NULL
   AND a.service_date >= p.start_date
GROUP BY 1,2,3,4,5,6

UNION ALL

-- ============================================
-- INSURANCE KPIs
-- ============================================
SELECT
    c.region_id,
    c.region_name,
    c.territory_id,
    c.territory_name,
    p.period_type,
    CASE
        WHEN latest_insurance_type ILIKE '%MEDICARE%'
            THEN 'AVLAYAH_Medicare_Patients'
        WHEN latest_insurance_type ILIKE '%MEDICAID%'
            THEN 'AVLAYAH_Medicaid_Patients'
        WHEN latest_insurance_type ILIKE '%COMMERCIAL%'
            THEN 'AVLAYAH_Commercial_Patients'
        ELSE 'AVLAYAH_Unknown_Patients'
    END AS metric_name,
    COUNT(DISTINCT patient_id) AS metric_value
FROM claims_base c
JOIN periods p
    ON p.start_date IS NOT NULL
   AND c.service_date >= p.start_date
GROUP BY 1,2,3,4,5,6

UNION ALL

-- ============================================
-- AGE KPIs
-- ============================================
SELECT
    c.region_id,
    c.region_name,
    c.territory_id,
    c.territory_name,
    p.period_type,
    CASE
        WHEN patient_age < 5 THEN 'AVLAYAH_PATIENTS_LT_5'
        WHEN patient_age BETWEEN 5 AND 10 THEN 'AVLAYAH_PATIENTS_5_10'
        WHEN patient_age BETWEEN 11 AND 16 THEN 'AVLAYAH_PATIENTS_11_16'
        ELSE 'AVLAYAH_PATIENTS_17_PLUS'
    END AS metric_name,
    COUNT(DISTINCT patient_id) AS metric_value
FROM claims_base c
JOIN periods p
    ON p.start_date IS NOT NULL
   AND c.service_date >= p.start_date
GROUP BY 1,2,3,4,5,6

UNION ALL

-- ============================================
-- SEVERITY KPIs
-- ============================================
SELECT
    c.region_id,
    c.region_name,
    c.territory_id,
    c.territory_name,
    p.period_type,
    CASE
        WHEN severity = 'Severe'
            THEN 'Severe_Avlayah'
        ELSE 'Attenuated_Avlayah'
    END AS metric_name,
    COUNT(DISTINCT patient_id) AS metric_value
FROM claims_base c
JOIN periods p
    ON p.start_date IS NOT NULL
   AND c.service_date >= p.start_date
GROUP BY 1,2,3,4,5,6

UNION ALL

-- ============================================
-- ELAPRASE KPIs
-- ============================================
SELECT
    a.primary_hcp_region_id_2yr       AS region_id,
    a.primary_hcp_region_2yr          AS region_name,
    a.primary_hcp_territory_id_2yr    AS territory_id,
    a.primary_hcp_territory_2yr       AS territory_name,
    p.period_type,
    CASE
        WHEN patient_age < 5 THEN 'ELAPRASE_LT_5'
        WHEN patient_age BETWEEN 5 AND 10 THEN 'ELAPRASE_5_10'
        WHEN patient_age BETWEEN 11 AND 16 THEN 'ELAPRASE_11_16'
        ELSE 'ELAPRASE_17_PLUS'
    END AS metric_name,
    COUNT(DISTINCT a.patient_id) AS metric_value
FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master a
JOIN com_edp_prd.cmpa_insights_internal_schema.MPSII_Treatment_claims b
    ON a.patient_id = b.patient_id
JOIN periods_elaprase p
    ON b.claim_date >= p.start_date
WHERE a.first_incidence_treatment_date IS NOT NULL
GROUP BY 1,2,3,4,5,6

UNION ALL

-- ============================================
-- HCP PRESCRIBED
-- ============================================
SELECT
    region_id,
    region_name,
    territory_id,
    territory_name,
    p.period_type,
    'HCP_PRESCRIBED',
    COUNT(DISTINCT npi)
FROM com_edp_prd.cmpa_insights_internal_schema.hcp_prescribed_avlayah h
JOIN periods p
    ON p.start_date IS NOT NULL
   AND h.service_date >= p.start_date
GROUP BY 1,2,3,4,5,6

UNION ALL

-- ============================================
-- HCO ORDERED
-- ============================================
SELECT
    region_id,
    region_name,
    territory_id,
    territory_name,
    p.period_type,
    'HCO_ORDERED',
    SUM(hco_accounts_ordered)
FROM com_edp_prd.cmpa_insights_internal_schema.hco_ordered_avlayah h
JOIN periods p
    ON p.start_date IS NOT NULL
   AND h.service_date >= p.start_date
GROUP BY 1,2,3,4,5,6

UNION ALL

-- ============================================
-- VIALS
-- ============================================
SELECT
    region_id,
    region_name,
    territory_id,
    territory_name,
    p.period_type,
    'TOTAL_VIALS',
    SUM(total_vials)
FROM com_edp_prd.cmpa_insights_internal_schema.vials_avlayah v
JOIN periods p
    ON p.start_date IS NOT NULL
   AND v.service_date >= p.start_date
GROUP BY 1,2,3,4,5,6

UNION ALL

SELECT
    region_id,
    region_name,
    territory_id,
    territory_name,
    p.period_type,
    'SP_VIALS',
    SUM(sp_vials)
FROM com_edp_prd.cmpa_insights_internal_schema.vials_avlayah v
JOIN periods p
    ON p.start_date IS NOT NULL
   AND v.service_date >= p.start_date
GROUP BY 1,2,3,4,5,6

UNION ALL

SELECT
    region_id,
    region_name,
    territory_id,
    territory_name,
    p.period_type,
    'HCO_VIALS',
    SUM(hco_vials)
FROM com_edp_prd.cmpa_insights_internal_schema.vials_avlayah v
JOIN periods p
    ON p.start_date IS NOT NULL
   AND v.service_date >= p.start_date
GROUP BY 1,2,3,4,5,6

UNION ALL

-- ============================================
-- CLAIMS / SP / HUB / CRM
-- ============================================
SELECT
    region_id,
    region_name,
    territory_id,
    territory_name,
    p.period_type,
    'Avlayah_Claims',
    COUNT(DISTINCT patient_id)
FROM claims_base c
JOIN periods p
    ON p.start_date IS NOT NULL
   AND c.service_date >= p.start_date
GROUP BY 1,2,3,4,5,6

UNION ALL

SELECT
    region_id,
    region_name,
    territory_id,
    territory_name,
    p.period_type,
    'Avlayah_SP',
    COUNT(DISTINCT patient_id)
FROM com_edp_prd.cmpa_insights_internal_schema.sp_patients_avlayah s
JOIN periods p
    ON p.start_date IS NOT NULL
   AND s.service_date >= p.start_date
GROUP BY 1,2,3,4,5,6

UNION ALL

SELECT
    region_id,
    region_name,
    territory_id,
    territory_name,
    p.period_type,
    'Avlayah_HUB',
    COUNT(DISTINCT patient_id)
FROM com_edp_prd.cmpa_insights_internal_schema.hub_patients_avlayah h
JOIN periods p
    ON p.start_date IS NOT NULL
   AND h.service_date >= p.start_date
GROUP BY 1,2,3,4,5,6

UNION ALL

SELECT
    region_id,
    region_name,
    territory_id,
    territory_name,
    p.period_type,
    'Avlayah_CRM',
    SUM(crm_patient_count)
FROM crm_dedup c
JOIN periods p
    ON p.start_date IS NOT NULL
   AND c.service_date >= p.start_date
GROUP BY 1,2,3,4,5,6
)

-- ============================================
-- FINAL GRID
-- ============================================
SELECT
    g.region_id,
    g.region_name,
    g.territory_id,
    g.territory_name,
    p.period_type,
    m.metric_name,
    COALESCE(SUM(f.metric_value),0) AS metric_value

FROM dim_geo g
CROSS JOIN dim_metrics m
CROSS JOIN periods p

LEFT JOIN fact_data f
    ON g.region_id = f.region_id
   AND g.territory_id = f.territory_id
   AND m.metric_name = f.metric_name
   AND p.period_type = f.period_type

GROUP BY 1,2,3,4,5,6;

In [0]:
-- CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.overview_aldashboard_metrics_period AS

-- WITH dim_geo AS (
--     SELECT DISTINCT
--         region_id,
--         region_name,
--         territory_id,
--         territory_name
--     FROM com_edp_prd.cmpa_insights_internal_schema.all_metrics_avlayah
-- ),

-- dim_metrics AS (
--     SELECT DISTINCT metric_name
--     FROM (
--         SELECT metric_name 
--         FROM com_edp_prd.cmpa_insights_internal_schema.all_metrics_avlayah

--         UNION ALL SELECT 'AVLAYAH_Medicare_Patients'
--         UNION ALL SELECT 'AVLAYAH_Medicaid_Patients'
--         UNION ALL SELECT 'AVLAYAH_Commercial_Patients'
--         UNION ALL SELECT 'AVLAYAH_Unknown_Patients'
--         UNION ALL SELECT 'AVLAYAH_PATIENTS_LT_5'
--         UNION ALL SELECT 'AVLAYAH_PATIENTS_5_10'
--         UNION ALL SELECT 'AVLAYAH_PATIENTS_11_16'
--         UNION ALL SELECT 'AVLAYAH_PATIENTS_17_PLUS'
--         UNION ALL SELECT 'Severe_Avlayah'
--         UNION ALL SELECT 'Attenuated_Avlayah'
--         UNION ALL SELECT 'ELAPRASE_LT_5'
--         UNION ALL SELECT 'ELAPRASE_5_10'
--         UNION ALL SELECT 'ELAPRASE_11_16'
--         UNION ALL SELECT 'ELAPRASE_17_PLUS'
--         UNION ALL SELECT 'HCP_PRESCRIBED'
--         UNION ALL SELECT 'HCO_ORDERED'
--         UNION ALL SELECT 'TOTAL_VIALS'
--         UNION ALL SELECT 'SP_VIALS'
--         UNION ALL SELECT 'HCO_VIALS'
--         UNION ALL SELECT 'Avlayah_Claims'
--         UNION ALL SELECT 'Avlayah_SP'
--         UNION ALL SELECT 'Avlayah_HUB'
--         UNION ALL SELECT 'Avlayah_CRM'
--     )
-- ),

-- dim_period AS (
--     SELECT 'LTD' AS period_type UNION
--     SELECT 'YTD' UNION
--     SELECT 'QTD' UNION
--     SELECT 'MTD'
-- ),

-- fact_data AS (
--     SELECT * FROM (
-- -- =========================
-- -- LTD
-- -- =========================
-- SELECT
--     region_id,
--     region_name,
--     territory_id,
--     territory_name,
--     'LTD' AS period_type,
--     metric_name,
--     COALESCE(SUM(metric_value), 0) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.all_metrics_avlayah
-- WHERE service_date >= DATE('2026-03-01')
-- GROUP BY 1,2,3,4,5,6

-- UNION ALL
-- SELECT
--     region_id,
--     region_name,
--     territory_id,
--     territory_name,
--     'LTD' AS period_type,
--     'AVLAYAH_Medicare_Patients' AS metric_name,
--     COUNT(DISTINCT patient_id) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.claims_patients_avlayah
-- WHERE service_date >= DATE('2026-03-01')
--   AND latest_insurance_type ILIKE '%MEDICARE%'
-- GROUP BY 1,2,3,4,5,6

-- UNION ALL
-- SELECT
--     region_id,
--     region_name,
--     territory_id,
--     territory_name,
--     'LTD' AS period_type,
--     'AVLAYAH_Medicaid_Patients' AS metric_name,
--     COUNT(DISTINCT patient_id) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.claims_patients_avlayah
-- WHERE service_date >= DATE('2026-03-01')
--   AND latest_insurance_type ILIKE '%MEDICAID%'
-- GROUP BY 1,2,3,4,5,6

-- UNION ALL
-- SELECT
--     region_id,
--     region_name,
--     territory_id,
--     territory_name,
--     'LTD' AS period_type,
--     'AVLAYAH_Commercial_Patients' AS metric_name,
--     COUNT(DISTINCT patient_id) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.claims_patients_avlayah
-- WHERE service_date >= DATE('2026-03-01')
--   AND latest_insurance_type ILIKE '%COMMERCIAL%'
-- GROUP BY 1,2,3,4,5,6

-- UNION ALL
-- SELECT
--     region_id,
--     region_name,
--     territory_id,
--     territory_name,
--     'LTD' AS period_type,
--     'AVLAYAH_Unknown_Patients' AS metric_name,
--     COUNT(DISTINCT patient_id) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.claims_patients_avlayah
-- WHERE service_date >= DATE('2026-03-01')
--   AND (
--         latest_insurance_type IS NULL
--         OR (
--             latest_insurance_type NOT ILIKE '%MEDICARE%'
--             AND latest_insurance_type NOT ILIKE '%MEDICAID%'
--             AND latest_insurance_type NOT ILIKE '%COMMERCIAL%'
--         )
--       )
-- GROUP BY 1,2,3,4,5,6

-- UNION ALL
-- SELECT
--     region_id,
--     region_name,
--     territory_id,
--     territory_name,
--     'LTD' AS period_type,
--     'AVLAYAH_PATIENTS_LT_5' AS metric_name,
--     COUNT(DISTINCT patient_id) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.claims_patients_avlayah
-- WHERE service_date >= DATE('2026-03-01')
--   AND patient_age < 5
-- GROUP BY 1,2,3,4,5,6

-- UNION ALL
-- SELECT
--     region_id,
--     region_name,
--     territory_id,
--     territory_name,
--     'LTD' AS period_type,
--     'AVLAYAH_PATIENTS_5_10' AS metric_name,
--     COUNT(DISTINCT patient_id) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.claims_patients_avlayah
-- WHERE service_date >= DATE('2026-03-01')
--   AND patient_age BETWEEN 5 AND 10
-- GROUP BY 1,2,3,4,5,6

-- UNION ALL
-- SELECT
--     region_id,
--     region_name,
--     territory_id,
--     territory_name,
--     'LTD' AS period_type,
--     'AVLAYAH_PATIENTS_11_16' AS metric_name,
--     COUNT(DISTINCT patient_id) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.claims_patients_avlayah
-- WHERE service_date >= DATE('2026-03-01')
--   AND patient_age BETWEEN 11 AND 16
-- GROUP BY 1,2,3,4,5,6

-- UNION ALL
-- SELECT
--     region_id,
--     region_name,
--     territory_id,
--     territory_name,
--     'LTD' AS period_type,
--     'AVLAYAH_PATIENTS_17_PLUS' AS metric_name,
--     COUNT(DISTINCT patient_id) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.claims_patients_avlayah
-- WHERE service_date >= DATE('2026-03-01')
--   AND patient_age >= 17
-- GROUP BY 1,2,3,4,5,6

-- UNION ALL
-- SELECT
--     region_id,
--     region_name,
--     territory_id,
--     territory_name,
--     'LTD' AS period_type,
--     'Severe_Avlayah' AS metric_name,
--     COUNT(DISTINCT patient_id) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.claims_patients_avlayah
-- WHERE service_date >= DATE('2026-03-01')
--   AND severity = 'Severe'
-- GROUP BY 1,2,3,4,5,6

-- UNION ALL
-- SELECT
--     region_id,
--     region_name,
--     territory_id,
--     territory_name,
--     'LTD' AS period_type,
--     'Attenuated_Avlayah' AS metric_name,
--     COUNT(DISTINCT patient_id) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.claims_patients_avlayah
-- WHERE service_date >= DATE('2026-03-01')
--   AND severity = 'Attenuated'
-- GROUP BY 1,2,3,4,5,6

-- UNION ALL
-- SELECT
--     a.primary_hcp_region_id_2yr,
--     a.primary_hcp_region_2yr,
--     a.primary_hcp_territory_id_2yr,
--     a.primary_hcp_territory_2yr,
--     'LTD' AS period_type,
--     'ELAPRASE_LT_5' AS metric_name,
--     COUNT(DISTINCT a.patient_id) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master a
-- LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.MPSII_Treatment_claims b
--     ON a.patient_id = b.patient_id
-- WHERE b.claim_date >= DATE('2023-08-01')
--   AND a.patient_age < 5
--   AND a.first_incidence_treatment_date IS NOT NULL
-- GROUP BY 1,2,3,4,5,6

-- UNION ALL
-- SELECT
--     a.primary_hcp_region_id_2yr,
--     a.primary_hcp_region_2yr,
--     a.primary_hcp_territory_id_2yr,
--     a.primary_hcp_territory_2yr,
--     'LTD' AS period_type,
--     'ELAPRASE_5_10' AS metric_name,
--     COUNT(DISTINCT a.patient_id) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master a
-- LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.MPSII_Treatment_claims b
--     ON a.patient_id = b.patient_id
-- WHERE b.claim_date >= DATE('2023-08-01')
--   AND a.patient_age BETWEEN 5 AND 10
--   AND a.first_incidence_treatment_date IS NOT NULL
-- GROUP BY 1,2,3,4,5,6

-- UNION ALL
-- SELECT
--     a.primary_hcp_region_id_2yr,
--     a.primary_hcp_region_2yr,
--     a.primary_hcp_territory_id_2yr,
--     a.primary_hcp_territory_2yr,
--     'LTD' AS period_type,
--     'ELAPRASE_11_16' AS metric_name,
--     COUNT(DISTINCT a.patient_id) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master a
-- LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.MPSII_Treatment_claims b
--     ON a.patient_id = b.patient_id
-- WHERE b.claim_date >= DATE('2023-08-01')
--   AND a.patient_age BETWEEN 11 AND 16
--   AND a.first_incidence_treatment_date IS NOT NULL
-- GROUP BY 1,2,3,4,5,6

-- UNION ALL
-- SELECT
--     a.primary_hcp_region_id_2yr,
--     a.primary_hcp_region_2yr,
--     a.primary_hcp_territory_id_2yr,
--     a.primary_hcp_territory_2yr,
--     'LTD' AS period_type,
--     'ELAPRASE_17_PLUS' AS metric_name,
--     COUNT(DISTINCT a.patient_id) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master a
-- LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.MPSII_Treatment_claims b
--     ON a.patient_id = b.patient_id
-- WHERE b.claim_date >= DATE('2023-08-01')
--   AND a.patient_age >= 17
--   AND a.first_incidence_treatment_date IS NOT NULL
-- GROUP BY 1,2,3,4,5,6

-- UNION ALL
-- SELECT
--     region_id,
--     region_name,
--     territory_id,
--     territory_name,
--     'LTD' AS period_type,
--     'HCP_PRESCRIBED' AS metric_name,
--     COALESCE(COUNT(DISTINCT npi), 0) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.hcp_prescribed_avlayah
-- WHERE service_date >= DATE('2026-03-01')
-- GROUP BY 1,2,3,4,5,6

-- UNION ALL
-- SELECT
--     region_id,
--     region_name,
--     territory_id,
--     territory_name,
--     'LTD' AS period_type,
--     'HCO_ORDERED' AS metric_name,
--     SUM(hco_accounts_ordered) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.hco_ordered_avlayah
-- WHERE service_date >= DATE('2026-03-01')
-- GROUP BY 1,2,3,4,5,6

-- UNION ALL
-- SELECT
--     region_id,
--     region_name,
--     territory_id,
--     territory_name,
--     'LTD' AS period_type,
--     'TOTAL_VIALS' AS metric_name,
--     COALESCE(SUM(total_vials), 0) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.vials_avlayah
-- WHERE service_date >= DATE('2025-01-01')
-- GROUP BY 1,2,3,4,5,6

-- UNION ALL
-- SELECT
--     region_id,
--     region_name,
--     territory_id,
--     territory_name,
--     'LTD' AS period_type,
--     'SP_VIALS' AS metric_name,
--     COALESCE(SUM(sp_vials), 0) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.vials_avlayah
-- WHERE service_date >= DATE('2025-01-01')
-- GROUP BY 1,2,3,4,5,6

-- UNION ALL
-- SELECT
--     region_id,
--     region_name,
--     territory_id,
--     territory_name,
--     'LTD' AS period_type,
--     'HCO_VIALS' AS metric_name,
--     COALESCE(SUM(HCO_VIALS), 0) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.vials_avlayah
-- WHERE service_date >= DATE('2025-01-01')
-- GROUP BY 1,2,3,4,5,6

-- -- Avlayah Claims
-- UNION ALL
-- SELECT
--     region_id,
--     region_name,
--     territory_id,
--     territory_name,
--     'LTD' AS period_type,
--     'Avlayah_Claims' AS metric_name,
--     COUNT(DISTINCT patient_id) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.claims_patients_avlayah
-- WHERE service_date >= DATE('2026-03-01')
-- GROUP BY 1,2,3,4,5,6

-- -- Avlayah SP
-- UNION ALL
-- SELECT
--     region_id,
--     region_name,
--     territory_id,
--     territory_name,
--     'LTD' AS period_type,
--     'Avlayah_SP' AS metric_name,
--     COUNT(DISTINCT patient_id) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.sp_patients_avlayah
-- WHERE service_date >= DATE('2026-03-01')
-- GROUP BY 1,2,3,4,5,6

-- -- Avlayah HUB
-- UNION ALL
-- SELECT
--     region_id,
--     region_name,
--     territory_id,
--     territory_name,
--     'LTD' AS period_type,
--     'Avlayah_HUB' AS metric_name,
--     COUNT(DISTINCT patient_id) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.hub_patients_avlayah
-- WHERE service_date >= DATE('2026-03-01')
-- GROUP BY 1,2,3,4,5,6

-- -- Avlayah CRM
-- UNION ALL
-- SELECT
--     region_id,
--     region_name,
--     territory_id,
--     territory_name,
--     'LTD' AS period_type,
--     'Avlayah_CRM' AS metric_name,
--     SUM(crm_patient_count) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.crm_patients_avlayah
-- WHERE service_date >= DATE('2026-03-01')
-- GROUP BY 1,2,3,4,5,6

-- UNION ALL

-- -- =========================
-- -- YTD
-- -- =========================
-- SELECT
--     region_id,
--     region_name,
--     territory_id,
--     territory_name,
--     'YTD' AS period_type,
--     metric_name,
--     COALESCE(SUM(metric_value), 0) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.all_metrics_avlayah
-- WHERE service_date >= DATE_TRUNC('year', (SELECT end_date FROM runtime_parameters))
-- GROUP BY 1,2,3,4,5,6

-- UNION ALL
-- SELECT
--     region_id,
--     region_name,
--     territory_id,
--     territory_name,
--     'YTD' AS period_type,
--     'AVLAYAH_Medicare_Patients' AS metric_name,
--     COUNT(DISTINCT patient_id) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.claims_patients_avlayah
-- WHERE service_date >= DATE_TRUNC('year', (SELECT end_date FROM runtime_parameters))
--   AND latest_insurance_type ILIKE '%MEDICARE%'
-- GROUP BY 1,2,3,4,5,6

-- UNION ALL
-- SELECT
--     region_id,
--     region_name,
--     territory_id,
--     territory_name,
--     'YTD' AS period_type,
--     'AVLAYAH_Medicaid_Patients' AS metric_name,
--     COUNT(DISTINCT patient_id) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.claims_patients_avlayah
-- WHERE service_date >= DATE_TRUNC('year', (SELECT end_date FROM runtime_parameters))
--   AND latest_insurance_type ILIKE '%MEDICAID%'
-- GROUP BY 1,2,3,4,5,6

-- UNION ALL
-- SELECT
--     region_id,
--     region_name,
--     territory_id,
--     territory_name,
--     'YTD' AS period_type,
--     'AVLAYAH_Commercial_Patients' AS metric_name,
--     COUNT(DISTINCT patient_id) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.claims_patients_avlayah
-- WHERE service_date >= DATE_TRUNC('year', (SELECT end_date FROM runtime_parameters))
--   AND latest_insurance_type ILIKE '%COMMERCIAL%'
-- GROUP BY 1,2,3,4,5,6

-- UNION ALL
-- SELECT
--     region_id,
--     region_name,
--     territory_id,
--     territory_name,
--     'YTD' AS period_type,
--     'AVLAYAH_Unknown_Patients' AS metric_name,
--     COUNT(DISTINCT patient_id) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.claims_patients_avlayah
-- WHERE service_date >= DATE_TRUNC('year', (SELECT end_date FROM runtime_parameters))
--   AND (
--         latest_insurance_type IS NULL
--         OR (
--             latest_insurance_type NOT ILIKE '%MEDICARE%'
--             AND latest_insurance_type NOT ILIKE '%MEDICAID%'
--             AND latest_insurance_type NOT ILIKE '%COMMERCIAL%'
--         )
--       )
-- GROUP BY 1,2,3,4,5,6

-- UNION ALL
-- SELECT
--     region_id,
--     region_name,
--     territory_id,
--     territory_name,
--     'YTD' AS period_type,
--     'AVLAYAH_PATIENTS_LT_5' AS metric_name,
--     COUNT(DISTINCT patient_id) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.claims_patients_avlayah
-- WHERE service_date >= DATE_TRUNC('year', (SELECT end_date FROM runtime_parameters))
--   AND patient_age < 5
-- GROUP BY 1,2,3,4,5,6

-- UNION ALL
-- SELECT
--     region_id,
--     region_name,
--     territory_id,
--     territory_name,
--     'YTD' AS period_type,
--     'AVLAYAH_PATIENTS_5_10' AS metric_name,
--     COUNT(DISTINCT patient_id) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.claims_patients_avlayah
-- WHERE service_date >= DATE_TRUNC('year', (SELECT end_date FROM runtime_parameters))
--   AND patient_age BETWEEN 5 AND 10
-- GROUP BY 1,2,3,4,5,6

-- UNION ALL
-- SELECT
--     region_id,
--     region_name,
--     territory_id,
--     territory_name,
--     'YTD' AS period_type,
--     'AVLAYAH_PATIENTS_11_16' AS metric_name,
--     COUNT(DISTINCT patient_id) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.claims_patients_avlayah
-- WHERE service_date >= DATE_TRUNC('year', (SELECT end_date FROM runtime_parameters))
--   AND patient_age BETWEEN 11 AND 16
-- GROUP BY 1,2,3,4,5,6

-- UNION ALL
-- SELECT
--     region_id,
--     region_name,
--     territory_id,
--     territory_name,
--     'YTD' AS period_type,
--     'AVLAYAH_PATIENTS_17_PLUS' AS metric_name,
--     COUNT(DISTINCT patient_id) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.claims_patients_avlayah
-- WHERE service_date >= DATE_TRUNC('year', (SELECT end_date FROM runtime_parameters))
--   AND patient_age >= 17
-- GROUP BY 1,2,3,4,5,6

-- UNION ALL
-- SELECT
--     region_id,
--     region_name,
--     territory_id,
--     territory_name,
--     'YTD' AS period_type,
--     'Severe_Avlayah' AS metric_name,
--     COUNT(DISTINCT patient_id) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.claims_patients_avlayah
-- WHERE service_date >= DATE_TRUNC('year', (SELECT end_date FROM runtime_parameters))
--   AND severity = 'Severe'
-- GROUP BY 1,2,3,4,5,6

-- UNION ALL
-- SELECT
--     region_id,
--     region_name,
--     territory_id,
--     territory_name,
--     'YTD' AS period_type,
--     'Attenuated_Avlayah' AS metric_name,
--     COUNT(DISTINCT patient_id) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.claims_patients_avlayah
-- WHERE service_date >= DATE_TRUNC('year', (SELECT end_date FROM runtime_parameters))
--   AND severity = 'Attenuated'
-- GROUP BY 1,2,3,4,5,6

-- UNION ALL
-- SELECT
--     a.primary_hcp_region_id_2yr,
--     a.primary_hcp_region_2yr,
--     a.primary_hcp_territory_id_2yr,
--     a.primary_hcp_territory_2yr,
--     'YTD' AS period_type,
--     'ELAPRASE_LT_5' AS metric_name,
--     COUNT(DISTINCT a.patient_id) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master a
-- LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.MPSII_Treatment_claims b
--     ON a.patient_id = b.patient_id
-- WHERE b.claim_date >= DATE_TRUNC('year', (SELECT end_date FROM runtime_parameters))
--   AND a.patient_age < 5
--   AND a.first_incidence_treatment_date IS NOT NULL
-- GROUP BY 1,2,3,4,5,6

-- UNION ALL
-- SELECT
--     a.primary_hcp_region_id_2yr,
--     a.primary_hcp_region_2yr,
--     a.primary_hcp_territory_id_2yr,
--     a.primary_hcp_territory_2yr,
--     'YTD' AS period_type,
--     'ELAPRASE_5_10' AS metric_name,
--     COUNT(DISTINCT a.patient_id) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master a
-- LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.MPSII_Treatment_claims b
--     ON a.patient_id = b.patient_id
-- WHERE b.claim_date >= DATE_TRUNC('year', (SELECT end_date FROM runtime_parameters))
--   AND a.patient_age BETWEEN 5 AND 10
--   AND a.first_incidence_treatment_date IS NOT NULL
-- GROUP BY 1,2,3,4,5,6

-- UNION ALL
-- SELECT
--     a.primary_hcp_region_id_2yr,
--     a.primary_hcp_region_2yr,
--     a.primary_hcp_territory_id_2yr,
--     a.primary_hcp_territory_2yr,
--     'YTD' AS period_type,
--     'ELAPRASE_11_16' AS metric_name,
--     COUNT(DISTINCT a.patient_id) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master a
-- LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.MPSII_Treatment_claims b
--     ON a.patient_id = b.patient_id
-- WHERE b.claim_date >= DATE_TRUNC('year', (SELECT end_date FROM runtime_parameters))
--   AND a.patient_age BETWEEN 11 AND 16
--   AND a.first_incidence_treatment_date IS NOT NULL
-- GROUP BY 1,2,3,4,5,6

-- UNION ALL
-- SELECT
--     a.primary_hcp_region_id_2yr,
--     a.primary_hcp_region_2yr,
--     a.primary_hcp_territory_id_2yr,
--     a.primary_hcp_territory_2yr,
--     'YTD' AS period_type,
--     'ELAPRASE_17_PLUS' AS metric_name,
--     COUNT(DISTINCT a.patient_id) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master a
-- LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.MPSII_Treatment_claims b
--     ON a.patient_id = b.patient_id
-- WHERE b.claim_date >= DATE_TRUNC('year', (SELECT end_date FROM runtime_parameters))
--   AND a.patient_age >= 17
--   AND a.first_incidence_treatment_date IS NOT NULL
-- GROUP BY 1,2,3,4,5,6

-- UNION ALL
-- SELECT
--     region_id,
--     region_name,
--     territory_id,
--     territory_name,
--     'YTD' AS period_type,
--     'HCP_PRESCRIBED' AS metric_name,
--     COALESCE(COUNT(DISTINCT npi), 0) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.hcp_prescribed_avlayah
-- WHERE service_date >= DATE_TRUNC('year', (SELECT end_date FROM runtime_parameters))
-- GROUP BY 1,2,3,4,5,6

-- UNION ALL
-- SELECT
--     region_id,
--     region_name,
--     territory_id,
--     territory_name,
--     'YTD' AS period_type,
--     'HCO_ORDERED' AS metric_name,
--     SUM(hco_accounts_ordered) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.hco_ordered_avlayah
-- WHERE service_date >= DATE_TRUNC('year', (SELECT end_date FROM runtime_parameters))
-- GROUP BY 1,2,3,4,5,6

-- UNION ALL
-- SELECT
--     region_id,
--     region_name,
--     territory_id,
--     territory_name,
--     'YTD' AS period_type,
--     'TOTAL_VIALS' AS metric_name,
--     COALESCE(SUM(total_vials), 0) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.vials_avlayah
-- WHERE service_date >= DATE_TRUNC('year', (SELECT end_date FROM runtime_parameters))
-- GROUP BY 1,2,3,4,5,6

-- UNION ALL
-- SELECT
--     region_id,
--     region_name,
--     territory_id,
--     territory_name,
--     'YTD' AS period_type,
--     'SP_VIALS' AS metric_name,
--     COALESCE(SUM(sp_vials), 0) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.vials_avlayah
-- WHERE service_date >= DATE_TRUNC('year', (SELECT end_date FROM runtime_parameters))
-- GROUP BY 1,2,3,4,5,6

-- UNION ALL
-- SELECT
--     region_id,
--     region_name,
--     territory_id,
--     territory_name,
--     'YTD' AS period_type,
--     'HCO_VIALS' AS metric_name,
--     COALESCE(SUM(HCO_VIALS), 0) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.vials_avlayah
-- WHERE service_date >= DATE_TRUNC('year', (SELECT end_date FROM runtime_parameters))
-- GROUP BY 1,2,3,4,5,6

-- UNION ALL
-- SELECT
--     region_id,
--     region_name,
--     territory_id,
--     territory_name,
--     'YTD' AS period_type,
--     'Avlayah_Claims' AS metric_name,
--     COUNT(DISTINCT patient_id) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.claims_patients_avlayah
-- WHERE service_date >= DATE_TRUNC('year', (SELECT end_date FROM runtime_parameters))
-- GROUP BY 1,2,3,4,5,6

-- -- Avlayah SP
-- UNION ALL
-- SELECT
--     region_id,
--     region_name,
--     territory_id,
--     territory_name,
--     'YTD' AS period_type,
--     'Avlayah_SP' AS metric_name,
--     COUNT(DISTINCT patient_id) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.sp_patients_avlayah
-- WHERE service_date >= DATE_TRUNC('year', (SELECT end_date FROM runtime_parameters))
-- GROUP BY 1,2,3,4,5,6

-- -- Avlayah HUB
-- UNION ALL
-- SELECT
--     region_id,
--     region_name,
--     territory_id,
--     territory_name,
--     'YTD' AS period_type,
--     'Avlayah_HUB' AS metric_name,
--     COUNT(DISTINCT patient_id) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.hub_patients_avlayah
-- WHERE service_date >= DATE_TRUNC('year', (SELECT end_date FROM runtime_parameters))
-- GROUP BY 1,2,3,4,5,6

-- -- Avlayah CRM
-- UNION ALL
-- SELECT
--     region_id,
--     region_name,
--     territory_id,
--     territory_name,
--     'YTD' AS period_type,
--     'Avlayah_CRM' AS metric_name,
--     SUM(crm_patient_count) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.crm_patients_avlayah
-- WHERE service_date >= DATE_TRUNC('year', (SELECT end_date FROM runtime_parameters))
-- GROUP BY 1,2,3,4,5,6

-- UNION ALL

-- -- =========================
-- -- QTD
-- -- =========================
-- SELECT
--     region_id,
--     region_name,
--     territory_id,
--     territory_name,
--     'QTD' AS period_type,
--     metric_name,
--     COALESCE(SUM(metric_value), 0) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.all_metrics_avlayah
-- WHERE service_date >= DATE_TRUNC('quarter', (SELECT end_date FROM runtime_parameters))
-- GROUP BY 1,2,3,4,5,6

-- UNION ALL
-- SELECT
--     region_id,
--     region_name,
--     territory_id,
--     territory_name,
--     'QTD' AS period_type,
--     'AVLAYAH_Medicare_Patients' AS metric_name,
--     COUNT(DISTINCT patient_id) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.claims_patients_avlayah
-- WHERE service_date >= DATE_TRUNC('quarter', (SELECT end_date FROM runtime_parameters))
--   AND latest_insurance_type ILIKE '%MEDICARE%'
-- GROUP BY 1,2,3,4,5,6

-- UNION ALL
-- SELECT
--     region_id,
--     region_name,
--     territory_id,
--     territory_name,
--     'QTD' AS period_type,
--     'AVLAYAH_Medicaid_Patients' AS metric_name,
--     COUNT(DISTINCT patient_id) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.claims_patients_avlayah
-- WHERE service_date >= DATE_TRUNC('quarter', (SELECT end_date FROM runtime_parameters))
--   AND latest_insurance_type ILIKE '%MEDICAID%'
-- GROUP BY 1,2,3,4,5,6

-- UNION ALL
-- SELECT
--     region_id,
--     region_name,
--     territory_id,
--     territory_name,
--     'QTD' AS period_type,
--     'AVLAYAH_Commercial_Patients' AS metric_name,
--     COUNT(DISTINCT patient_id) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.claims_patients_avlayah
-- WHERE service_date >= DATE_TRUNC('quarter', (SELECT end_date FROM runtime_parameters))
--   AND latest_insurance_type ILIKE '%COMMERCIAL%'
-- GROUP BY 1,2,3,4,5,6

-- UNION ALL
-- SELECT
--     region_id,
--     region_name,
--     territory_id,
--     territory_name,
--     'QTD' AS period_type,
--     'AVLAYAH_Unknown_Patients' AS metric_name,
--     COUNT(DISTINCT patient_id) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.claims_patients_avlayah
-- WHERE service_date >= DATE_TRUNC('quarter', (SELECT end_date FROM runtime_parameters))
--   AND (
--         latest_insurance_type IS NULL
--         OR (
--             latest_insurance_type NOT ILIKE '%MEDICARE%'
--             AND latest_insurance_type NOT ILIKE '%MEDICAID%'
--             AND latest_insurance_type NOT ILIKE '%COMMERCIAL%'
--         )
--       )
-- GROUP BY 1,2,3,4,5,6

-- UNION ALL
-- SELECT
--     region_id,
--     region_name,
--     territory_id,
--     territory_name,
--     'QTD' AS period_type,
--     'AVLAYAH_PATIENTS_LT_5' AS metric_name,
--     COUNT(DISTINCT patient_id) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.claims_patients_avlayah
-- WHERE service_date >= DATE_TRUNC('quarter', (SELECT end_date FROM runtime_parameters))
--   AND patient_age < 5
-- GROUP BY 1,2,3,4,5,6

-- UNION ALL
-- SELECT
--     region_id,
--     region_name,
--     territory_id,
--     territory_name,
--     'QTD' AS period_type,
--     'AVLAYAH_PATIENTS_5_10' AS metric_name,
--     COUNT(DISTINCT patient_id) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.claims_patients_avlayah
-- WHERE service_date >= DATE_TRUNC('quarter', (SELECT end_date FROM runtime_parameters))
--   AND patient_age BETWEEN 5 AND 10
-- GROUP BY 1,2,3,4,5,6

-- UNION ALL
-- SELECT
--     region_id,
--     region_name,
--     territory_id,
--     territory_name,
--     'QTD' AS period_type,
--     'AVLAYAH_PATIENTS_11_16' AS metric_name,
--     COUNT(DISTINCT patient_id) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.claims_patients_avlayah
-- WHERE service_date >= DATE_TRUNC('quarter', (SELECT end_date FROM runtime_parameters))
--   AND patient_age BETWEEN 11 AND 16
-- GROUP BY 1,2,3,4,5,6

-- UNION ALL
-- SELECT
--     region_id,
--     region_name,
--     territory_id,
--     territory_name,
--     'QTD' AS period_type,
--     'AVLAYAH_PATIENTS_17_PLUS' AS metric_name,
--     COUNT(DISTINCT patient_id) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.claims_patients_avlayah
-- WHERE service_date >= DATE_TRUNC('quarter', (SELECT end_date FROM runtime_parameters))
--   AND patient_age >= 17
-- GROUP BY 1,2,3,4,5,6

-- UNION ALL
-- SELECT
--     region_id,
--     region_name,
--     territory_id,
--     territory_name,
--     'QTD' AS period_type,
--     'Severe_Avlayah' AS metric_name,
--     COUNT(DISTINCT patient_id) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.claims_patients_avlayah
-- WHERE service_date >= DATE_TRUNC('quarter', (SELECT end_date FROM runtime_parameters))
--   AND severity = 'Severe'
-- GROUP BY 1,2,3,4,5,6

-- UNION ALL
-- SELECT
--     region_id,
--     region_name,
--     territory_id,
--     territory_name,
--     'QTD' AS period_type,
--     'Attenuated_Avlayah' AS metric_name,
--     COUNT(DISTINCT patient_id) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.claims_patients_avlayah
-- WHERE service_date >= DATE_TRUNC('quarter', (SELECT end_date FROM runtime_parameters))
--   AND severity = 'Attenuated'
-- GROUP BY 1,2,3,4,5,6

-- UNION ALL
-- SELECT
--     a.primary_hcp_region_id_2yr,
--     a.primary_hcp_region_2yr,
--     a.primary_hcp_territory_id_2yr,
--     a.primary_hcp_territory_2yr,
--     'QTD' AS period_type,
--     'ELAPRASE_LT_5' AS metric_name,
--     COUNT(DISTINCT a.patient_id) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master a
-- LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.MPSII_Treatment_claims b
--     ON a.patient_id = b.patient_id
-- WHERE b.claim_date >= DATE_TRUNC('quarter', (SELECT end_date FROM runtime_parameters))
--   AND a.patient_age < 5
--   AND a.first_incidence_treatment_date IS NOT NULL
-- GROUP BY 1,2,3,4,5,6

-- UNION ALL
-- SELECT
--     a.primary_hcp_region_id_2yr,
--     a.primary_hcp_region_2yr,
--     a.primary_hcp_territory_id_2yr,
--     a.primary_hcp_territory_2yr,
--     'QTD' AS period_type,
--     'ELAPRASE_5_10' AS metric_name,
--     COUNT(DISTINCT a.patient_id) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master a
-- LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.MPSII_Treatment_claims b
--     ON a.patient_id = b.patient_id
-- WHERE b.claim_date >= DATE_TRUNC('quarter', (SELECT end_date FROM runtime_parameters))
--   AND a.patient_age BETWEEN 5 AND 10
--   AND a.first_incidence_treatment_date IS NOT NULL
-- GROUP BY 1,2,3,4,5,6

-- UNION ALL
-- SELECT
--     a.primary_hcp_region_id_2yr,
--     a.primary_hcp_region_2yr,
--     a.primary_hcp_territory_id_2yr,
--     a.primary_hcp_territory_2yr,
--     'QTD' AS period_type,
--     'ELAPRASE_11_16' AS metric_name,
--     COUNT(DISTINCT a.patient_id) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master a
-- LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.MPSII_Treatment_claims b
--     ON a.patient_id = b.patient_id
-- WHERE b.claim_date >= DATE_TRUNC('quarter', (SELECT end_date FROM runtime_parameters))
--   AND a.patient_age BETWEEN 11 AND 16
--   AND a.first_incidence_treatment_date IS NOT NULL
-- GROUP BY 1,2,3,4,5,6

-- UNION ALL
-- SELECT
--     a.primary_hcp_region_id_2yr,
--     a.primary_hcp_region_2yr,
--     a.primary_hcp_territory_id_2yr,
--     a.primary_hcp_territory_2yr,
--     'QTD' AS period_type,
--     'ELAPRASE_17_PLUS' AS metric_name,
--     COUNT(DISTINCT a.patient_id) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master a
-- LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.MPSII_Treatment_claims b
--     ON a.patient_id = b.patient_id
-- WHERE b.claim_date >= DATE_TRUNC('quarter', (SELECT end_date FROM runtime_parameters))
--   AND a.patient_age >= 17
--   AND a.first_incidence_treatment_date IS NOT NULL
-- GROUP BY 1,2,3,4,5,6

-- UNION ALL
-- SELECT
--     region_id,
--     region_name,
--     territory_id,
--     territory_name,
--     'QTD' AS period_type,
--     'HCP_PRESCRIBED' AS metric_name,
--     COALESCE(COUNT(DISTINCT npi), 0) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.hcp_prescribed_avlayah
-- WHERE service_date >= DATE_TRUNC('quarter', (SELECT end_date FROM runtime_parameters))
-- GROUP BY 1,2,3,4,5,6

-- UNION ALL
-- SELECT
--     region_id,
--     region_name,
--     territory_id,
--     territory_name,
--     'QTD' AS period_type,
--     'HCO_ORDERED' AS metric_name,
--     SUM(hco_accounts_ordered) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.hco_ordered_avlayah
-- WHERE service_date >= DATE_TRUNC('quarter', (SELECT end_date FROM runtime_parameters))
-- GROUP BY 1,2,3,4,5,6

-- UNION ALL
-- SELECT
--     region_id,
--     region_name,
--     territory_id,
--     territory_name,
--     'QTD' AS period_type,
--     'TOTAL_VIALS' AS metric_name,
--     COALESCE(SUM(total_vials), 0) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.vials_avlayah
-- WHERE service_date >= DATE_TRUNC('quarter', (SELECT end_date FROM runtime_parameters))
-- GROUP BY 1,2,3,4,5,6

-- UNION ALL
-- SELECT
--     region_id,
--     region_name,
--     territory_id,
--     territory_name,
--     'QTD' AS period_type,
--     'SP_VIALS' AS metric_name,
--     COALESCE(SUM(sp_vials), 0) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.vials_avlayah
-- WHERE service_date >= DATE_TRUNC('quarter', (SELECT end_date FROM runtime_parameters))
-- GROUP BY 1,2,3,4,5,6

-- UNION ALL
-- SELECT
--     region_id,
--     region_name,
--     territory_id,
--     territory_name,
--     'QTD' AS period_type,
--     'HCO_VIALS' AS metric_name,
--     COALESCE(SUM(HCO_VIALS), 0) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.vials_avlayah
-- WHERE service_date >= DATE_TRUNC('quarter', (SELECT end_date FROM runtime_parameters))
-- GROUP BY 1,2,3,4,5,6

-- -- Avlayah Claims
-- UNION ALL
-- SELECT
--     region_id,
--     region_name,
--     territory_id,
--     territory_name,
--     'QTD' AS period_type,
--     'Avlayah_Claims' AS metric_name,
--     COUNT(DISTINCT patient_id) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.claims_patients_avlayah
-- WHERE service_date >= DATE_TRUNC('quarter', (SELECT end_date FROM runtime_parameters))
-- GROUP BY 1,2,3,4,5,6

-- -- Avlayah SP
-- UNION ALL
-- SELECT
--     region_id,
--     region_name,
--     territory_id,
--     territory_name,
--     'QTD' AS period_type,
--     'Avlayah_SP' AS metric_name,
--     COUNT(DISTINCT patient_id) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.sp_patients_avlayah
-- WHERE service_date >= DATE_TRUNC('quarter', (SELECT end_date FROM runtime_parameters))
-- GROUP BY 1,2,3,4,5,6

-- -- Avlayah HUB
-- UNION ALL
-- SELECT
--     region_id,
--     region_name,
--     territory_id,
--     territory_name,
--     'QTD' AS period_type,
--     'Avlayah_HUB' AS metric_name,
--     COUNT(DISTINCT patient_id) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.hub_patients_avlayah
-- WHERE service_date >= DATE_TRUNC('quarter', (SELECT end_date FROM runtime_parameters))
-- GROUP BY 1,2,3,4,5,6

-- -- Avlayah CRM
-- UNION ALL
-- SELECT
--     region_id,
--     region_name,
--     territory_id,
--     territory_name,
--     'QTD' AS period_type,
--     'Avlayah_CRM' AS metric_name,
--     SUM(crm_patient_count) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.crm_patients_avlayah
-- WHERE service_date >= DATE_TRUNC('quarter', (SELECT end_date FROM runtime_parameters))
-- GROUP BY 1,2,3,4,5,6

-- UNION ALL

-- -- =========================
-- -- MTD
-- -- =========================
-- SELECT
--     region_id,
--     region_name,
--     territory_id,
--     territory_name,
--     'MTD' AS period_type,
--     metric_name,
--     COALESCE(SUM(metric_value), 0) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.all_metrics_avlayah
-- WHERE service_date >= DATE_TRUNC('month', (SELECT end_date FROM runtime_parameters))
-- GROUP BY 1,2,3,4,5,6

-- UNION ALL
-- SELECT
--     region_id,
--     region_name,
--     territory_id,
--     territory_name,
--     'MTD' AS period_type,
--     'AVLAYAH_Medicare_Patients' AS metric_name,
--     COUNT(DISTINCT patient_id) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.claims_patients_avlayah
-- WHERE service_date >= DATE_TRUNC('month', (SELECT end_date FROM runtime_parameters))
--   AND latest_insurance_type ILIKE '%MEDICARE%'
-- GROUP BY 1,2,3,4,5,6

-- UNION ALL
-- SELECT
--     region_id,
--     region_name,
--     territory_id,
--     territory_name,
--     'MTD' AS period_type,
--     'AVLAYAH_Medicaid_Patients' AS metric_name,
--     COUNT(DISTINCT patient_id) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.claims_patients_avlayah
-- WHERE service_date >= DATE_TRUNC('month', (SELECT end_date FROM runtime_parameters))
--   AND latest_insurance_type ILIKE '%MEDICAID%'
-- GROUP BY 1,2,3,4,5,6

-- UNION ALL
-- SELECT
--     region_id,
--     region_name,
--     territory_id,
--     territory_name,
--     'MTD' AS period_type,
--     'AVLAYAH_Commercial_Patients' AS metric_name,
--     COUNT(DISTINCT patient_id) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.claims_patients_avlayah
-- WHERE service_date >= DATE_TRUNC('month', (SELECT end_date FROM runtime_parameters))
--   AND latest_insurance_type ILIKE '%COMMERCIAL%'
-- GROUP BY 1,2,3,4,5,6

-- UNION ALL
-- SELECT
--     region_id,
--     region_name,
--     territory_id,
--     territory_name,
--     'MTD' AS period_type,
--     'AVLAYAH_Unknown_Patients' AS metric_name,
--     COUNT(DISTINCT patient_id) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.claims_patients_avlayah
-- WHERE service_date >= DATE_TRUNC('month', (SELECT end_date FROM runtime_parameters))
--   AND (
--         latest_insurance_type IS NULL
--         OR (
--             latest_insurance_type NOT ILIKE '%MEDICARE%'
--             AND latest_insurance_type NOT ILIKE '%MEDICAID%'
--             AND latest_insurance_type NOT ILIKE '%COMMERCIAL%'
--         )
--       )
-- GROUP BY 1,2,3,4,5,6

-- UNION ALL
-- SELECT
--     region_id,
--     region_name,
--     territory_id,
--     territory_name,
--     'MTD' AS period_type,
--     'AVLAYAH_PATIENTS_LT_5' AS metric_name,
--     COUNT(DISTINCT patient_id) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.claims_patients_avlayah
-- WHERE service_date >= DATE_TRUNC('month', (SELECT end_date FROM runtime_parameters))
--   AND patient_age < 5
-- GROUP BY 1,2,3,4,5,6

-- UNION ALL
-- SELECT
--     region_id,
--     region_name,
--     territory_id,
--     territory_name,
--     'MTD' AS period_type,
--     'AVLAYAH_PATIENTS_5_10' AS metric_name,
--     COUNT(DISTINCT patient_id) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.claims_patients_avlayah
-- WHERE service_date >= DATE_TRUNC('month', (SELECT end_date FROM runtime_parameters))
--   AND patient_age BETWEEN 5 AND 10
-- GROUP BY 1,2,3,4,5,6

-- UNION ALL
-- SELECT
--     region_id,
--     region_name,
--     territory_id,
--     territory_name,
--     'MTD' AS period_type,
--     'AVLAYAH_PATIENTS_11_16' AS metric_name,
--     COUNT(DISTINCT patient_id) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.claims_patients_avlayah
-- WHERE service_date >= DATE_TRUNC('month', (SELECT end_date FROM runtime_parameters))
--   AND patient_age BETWEEN 11 AND 16
-- GROUP BY 1,2,3,4,5,6

-- UNION ALL
-- SELECT
--     region_id,
--     region_name,
--     territory_id,
--     territory_name,
--     'MTD' AS period_type,
--     'AVLAYAH_PATIENTS_17_PLUS' AS metric_name,
--     COUNT(DISTINCT patient_id) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.claims_patients_avlayah
-- WHERE service_date >= DATE_TRUNC('month', (SELECT end_date FROM runtime_parameters))
--   AND patient_age >= 17
-- GROUP BY 1,2,3,4,5,6

-- UNION ALL
-- SELECT
--     region_id,
--     region_name,
--     territory_id,
--     territory_name,
--     'MTD' AS period_type,
--     'Severe_Avlayah' AS metric_name,
--     COUNT(DISTINCT patient_id) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.claims_patients_avlayah
-- WHERE service_date >= DATE_TRUNC('month', (SELECT end_date FROM runtime_parameters))
--   AND severity = 'Severe'
-- GROUP BY 1,2,3,4,5,6

-- UNION ALL
-- SELECT
--     region_id,
--     region_name,
--     territory_id,
--     territory_name,
--     'MTD' AS period_type,
--     'Attenuated_Avlayah' AS metric_name,
--     COUNT(DISTINCT patient_id) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.claims_patients_avlayah
-- WHERE service_date >= DATE_TRUNC('month', (SELECT end_date FROM runtime_parameters))
--   AND severity = 'Attenuated'
-- GROUP BY 1,2,3,4,5,6

-- UNION ALL
-- SELECT
--     a.primary_hcp_region_id_2yr,
--     a.primary_hcp_region_2yr,
--     a.primary_hcp_territory_id_2yr,
--     a.primary_hcp_territory_2yr,
--     'MTD' AS period_type,
--     'ELAPRASE_LT_5' AS metric_name,
--     COUNT(DISTINCT a.patient_id) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master a
-- LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.MPSII_Treatment_claims b
--     ON a.patient_id = b.patient_id
-- WHERE b.claim_date >= DATE_TRUNC('month', (SELECT end_date FROM runtime_parameters))
--   AND a.patient_age < 5
--   AND a.first_incidence_treatment_date IS NOT NULL
-- GROUP BY 1,2,3,4,5,6

-- UNION ALL
-- SELECT
--     a.primary_hcp_region_id_2yr,
--     a.primary_hcp_region_2yr,
--     a.primary_hcp_territory_id_2yr,
--     a.primary_hcp_territory_2yr,
--     'MTD' AS period_type,
--     'ELAPRASE_5_10' AS metric_name,
--     COUNT(DISTINCT a.patient_id) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master a
-- LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.MPSII_Treatment_claims b
--     ON a.patient_id = b.patient_id
-- WHERE b.claim_date >= DATE_TRUNC('month', (SELECT end_date FROM runtime_parameters))
--   AND a.patient_age BETWEEN 5 AND 10
--   AND a.first_incidence_treatment_date IS NOT NULL
-- GROUP BY 1,2,3,4,5,6

-- UNION ALL
-- SELECT
--     a.primary_hcp_region_id_2yr,
--     a.primary_hcp_region_2yr,
--     a.primary_hcp_territory_id_2yr,
--     a.primary_hcp_territory_2yr,
--     'MTD' AS period_type,
--     'ELAPRASE_11_16' AS metric_name,
--     COUNT(DISTINCT a.patient_id) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master a
-- LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.MPSII_Treatment_claims b
--     ON a.patient_id = b.patient_id
-- WHERE b.claim_date >= DATE_TRUNC('month', (SELECT end_date FROM runtime_parameters))
--   AND a.patient_age BETWEEN 11 AND 16
--   AND a.first_incidence_treatment_date IS NOT NULL
-- GROUP BY 1,2,3,4,5,6

-- UNION ALL
-- SELECT
--     a.primary_hcp_region_id_2yr,
--     a.primary_hcp_region_2yr,
--     a.primary_hcp_territory_id_2yr,
--     a.primary_hcp_territory_2yr,
--     'MTD' AS period_type,
--     'ELAPRASE_17_PLUS' AS metric_name,
--     COUNT(DISTINCT a.patient_id) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master a
-- LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.MPSII_Treatment_claims b
--     ON a.patient_id = b.patient_id
-- WHERE b.claim_date >= DATE_TRUNC('month', (SELECT end_date FROM runtime_parameters))
--   AND a.patient_age >= 17
--   AND a.first_incidence_treatment_date IS NOT NULL
-- GROUP BY 1,2,3,4,5,6

-- UNION ALL
-- SELECT
--     region_id,
--     region_name,
--     territory_id,
--     territory_name,
--     'MTD' AS period_type,
--     'HCP_PRESCRIBED' AS metric_name,
--     COALESCE(COUNT(DISTINCT npi), 0) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.hcp_prescribed_avlayah
-- WHERE service_date >= DATE_TRUNC('month', (SELECT end_date FROM runtime_parameters))
-- GROUP BY 1,2,3,4,5,6

-- UNION ALL
-- SELECT
--     region_id,
--     region_name,
--     territory_id,
--     territory_name,
--     'MTD' AS period_type,
--     'HCO_ORDERED' AS metric_name,
--     SUM(hco_accounts_ordered) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.hco_ordered_avlayah
-- WHERE service_date >= DATE_TRUNC('month', (SELECT end_date FROM runtime_parameters))
-- GROUP BY 1,2,3,4,5,6

-- UNION ALL
-- SELECT
--     region_id,
--     region_name,
--     territory_id,
--     territory_name,
--     'MTD' AS period_type,
--     'TOTAL_VIALS' AS metric_name,
--     COALESCE(SUM(total_vials), 0) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.vials_avlayah
-- WHERE service_date >= DATE_TRUNC('month', (SELECT end_date FROM runtime_parameters))
-- GROUP BY 1,2,3,4,5,6

-- UNION ALL
-- SELECT
--     region_id,
--     region_name,
--     territory_id,
--     territory_name,
--     'MTD' AS period_type,
--     'SP_VIALS' AS metric_name,
--     COALESCE(SUM(sp_vials), 0) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.vials_avlayah
-- WHERE service_date >= DATE_TRUNC('month', (SELECT end_date FROM runtime_parameters))
-- GROUP BY 1,2,3,4,5,6

-- UNION ALL
-- SELECT
--     region_id,
--     region_name,
--     territory_id,
--     territory_name,
--     'MTD' AS period_type,
--     'HCO_VIALS' AS metric_name,
--     COALESCE(SUM(HCO_VIALS), 0) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.vials_avlayah
-- WHERE service_date >= DATE_TRUNC('month', (SELECT end_date FROM runtime_parameters))
-- GROUP BY 1,2,3,4,5,6

-- -- UNION ALL
-- -- SELECT
-- --     region_id,
-- --     region_name,
-- --     territory_id,
-- --     territory_name,
-- --     'MTD' AS period_type,
-- --     'SD_VIALS' AS metric_name,
-- --     COALESCE(SUM(sd_vials), 0) AS metric_value
-- -- FROM com_edp_prd.cmpa_insights_internal_schema.vials_avlayah
-- -- WHERE service_date >= DATE_TRUNC('month', (SELECT end_date FROM runtime_parameters))
-- -- GROUP BY 1,2,3,4,5,6
-- -- Avlayah Claims
-- UNION ALL
-- SELECT
--     region_id,
--     region_name,
--     territory_id,
--     territory_name,
--     'MTD' AS period_type,
--     'Avlayah_Claims' AS metric_name,
--     COUNT(DISTINCT patient_id) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.claims_patients_avlayah
-- WHERE service_date >= DATE_TRUNC('month', (SELECT end_date FROM runtime_parameters))
-- GROUP BY 1,2,3,4,5,6

-- -- Avlayah SP
-- UNION ALL
-- SELECT
--     region_id,
--     region_name,
--     territory_id,
--     territory_name,
--     'MTD' AS period_type,
--     'Avlayah_SP' AS metric_name,
--     COUNT(DISTINCT patient_id) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.sp_patients_avlayah
-- WHERE service_date >= DATE_TRUNC('month', (SELECT end_date FROM runtime_parameters))
-- GROUP BY 1,2,3,4,5,6

-- -- Avlayah HUB
-- UNION ALL
-- SELECT
--     region_id,
--     region_name,
--     territory_id,
--     territory_name,
--     'MTD' AS period_type,
--     'Avlayah_HUB' AS metric_name,
--     COUNT(DISTINCT patient_id) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.hub_patients_avlayah
-- WHERE service_date >= DATE_TRUNC('month', (SELECT end_date FROM runtime_parameters))
-- GROUP BY 1,2,3,4,5,6

-- -- Avlayah CRM
-- UNION ALL
-- SELECT
--     region_id,
--     region_name,
--     territory_id,
--     territory_name,
--     'MTD' AS period_type,
--     'Avlayah_CRM' AS metric_name,
--     SUM(crm_patient_count) AS metric_value
-- FROM com_edp_prd.cmpa_insights_internal_schema.crm_patients_avlayah
-- WHERE service_date >= DATE_TRUNC('month', (SELECT end_date FROM runtime_parameters))
-- GROUP BY 1,2,3,4,5,6


--     )
-- )

-- SELECT
--     g.region_id,
--     g.region_name,
--     g.territory_id,
--     g.territory_name,
--     p.period_type,
--     CASE 
--     WHEN m.metric_name = 'CLAIMS_PATIENTS' THEN 'Avlayah_Claims'
--     WHEN m.metric_name = 'SP_PATIENTS' THEN 'Avlayah_SP'
--     WHEN m.metric_name = 'HUB_PATIENTS' THEN 'Avlayah_HUB'
--     WHEN m.metric_name = 'CRM_PATIENTS' THEN 'Avlayah_CRM'
--     ELSE m.metric_name
--     END AS metric_name,
--     COALESCE(SUM(f.metric_value), 0) AS metric_value

-- FROM dim_geo g
-- CROSS JOIN dim_metrics m
-- CROSS JOIN dim_period p

-- LEFT JOIN fact_data f
--     ON g.region_id = f.region_id
--     AND g.territory_id = f.territory_id
--     AND m.metric_name = f.metric_name
--     AND p.period_type = f.period_type

--     GROUP BY
--     g.region_id,
--     g.region_name,
--     g.territory_id,
--     g.territory_name,
--     p.period_type,
--     m.metric_name;

In [0]:
CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.overview_aldashboard_metrics_period AS
select
'ACCOUNT_LEADS' as dashboard_name,
'OVERVIEW' as section_name,
* FROM com_edp_prd.cmpa_insights_internal_schema.overview_aldashboard_metrics_period

In [0]:
-- SELECT 
--     SUM(total_vials)
-- FROM com_edp_prd.cmpa_insights_internal_schema.vials_avlayah;

with shipments_dedup AS (
    SELECT *
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (
                   PARTITION BY crx_account_id, invoicedate, ndc, quantity_shipped
                   ORDER BY ingestion_date DESC
               ) AS rn
        FROM com_edp_prd.com_intgr.distribution_sd_shipments
        WHERE ndc = '84976-0001-01'   
    )
    WHERE rn = 1
)

SELECT 
    SUM(quantity_shipped) AS raw_qty
FROM shipments_dedup;

In [0]:
SELECT * FROM com_edp_prd.cmpa_insights_internal_schema.overview_aldashboard_metrics_period

In [0]:
SELECT distinct metric_name FROM com_edp_prd.cmpa_insights_internal_schema.overview_aldashboard_metrics_period

In [0]:
%sql
CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.HCO_Ordered_Account_Level AS

WITH shipments_dedup AS (
    SELECT *
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (
                   PARTITION BY invoice_number
                   ORDER BY ingestion_date DESC
               ) AS rn
        FROM com_edp_prd.com_intgr.distribution_sd_shipments
        WHERE is_current = true
          AND ndc = '84976-0001-01'
    )
    WHERE rn = 1
),

accounts_dedup AS (
    SELECT *
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (
                   PARTITION BY crx_account_id
                   ORDER BY ingestion_date DESC
               ) AS rn
        FROM com_edp_prd.com_intgr.distribution_accounts
        WHERE is_current = true
    )
    WHERE rn = 1
),

zip_territory AS (
    SELECT DISTINCT
        zipcode,
        territory_id,
        territory_name,
        region_id,
        region_name
    FROM com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping
),

base AS (
    SELECT
        s.crx_account_id,
        a.account_facility_name,
        a.account_type,
        CAST(s.order_date AS DATE) AS order_date,
        s.quantity_shipped,
        s.ship_to_address_postal_code,
        z.territory_id,
        z.territory_name,
        z.region_id,
        z.region_name
    FROM shipments_dedup s
    LEFT JOIN accounts_dedup a
        ON s.crx_account_id = a.crx_account_id
    LEFT JOIN zip_territory z
        ON SUBSTRING(s.ship_to_address_postal_code, 1, 5) = z.zipcode
)

SELECT
    crx_account_id,
    account_facility_name,

    CASE 
        WHEN UPPER(account_type) = 'SP'
        OR UPPER(account_facility_name) LIKE '%ORSINI%' THEN 'SP'
        ELSE 'HCO'
    END AS account_type,

    SUBSTRING(ship_to_address_postal_code, 1, 5) AS postal_code,

    territory_id,
    territory_name,
    region_id,
    region_name,

    /* LTD from March 1, 2026 */
    SUM(CASE 
        WHEN order_date >= DATE('2026-03-01') 
        THEN quantity_shipped 
        ELSE 0
    END) AS LTD,

    /* MTD */
    SUM(CASE 
        WHEN order_date >= DATE_TRUNC('month', CURRENT_DATE) 
        THEN quantity_shipped 
        ELSE 0
    END) AS MTD,

    /* QTD */
    SUM(CASE 
        WHEN order_date >= DATE_TRUNC('quarter', CURRENT_DATE) 
        THEN quantity_shipped 
        ELSE 0
    END) AS QTD,

    /* YTD */
    SUM(CASE 
        WHEN order_date >= DATE_TRUNC('year', CURRENT_DATE) 
        THEN quantity_shipped 
        ELSE 0
    END) AS YTD,

    /* WTD START DATE */
    DATE_SUB(CURRENT_DATE(), DAYOFWEEK(CURRENT_DATE()) + 6) AS WTD_START_DATE,

    /* WTD END DATE */
    DATE_SUB(CURRENT_DATE(), DAYOFWEEK(CURRENT_DATE())) AS WTD_END_DATE,

    /* WTD */
    SUM(CASE 
        WHEN order_date BETWEEN
             DATE_SUB(CURRENT_DATE(), DAYOFWEEK(CURRENT_DATE()) + 6)
         AND DATE_SUB(CURRENT_DATE(), DAYOFWEEK(CURRENT_DATE()))
        THEN quantity_shipped
        ELSE 0
    END) AS WTD

FROM base

GROUP BY
    crx_account_id,
    account_type,
    account_facility_name,
    CASE 
        WHEN UPPER(account_facility_name) LIKE '%ORSINI%' THEN 'SP'
        ELSE 'HCO'
    END,
    SUBSTRING(ship_to_address_postal_code, 1, 5),
    territory_id,
    territory_name,
    region_id,
    region_name;

In [0]:
select * from cmpa_insights_internal_schema.HCO_Ordered_Account_Level

In [0]:
WITH q1 AS (

    SELECT DISTINCT
        crx_account_id
    FROM (

        SELECT
            s.crx_account_id,

            ROW_NUMBER() OVER (
                PARTITION BY s.crx_account_id
                ORDER BY s.order_date ASC
            ) AS rn,

            CASE 
                WHEN UPPER(a.account_type) = 'SP'
                     OR UPPER(a.account_facility_name) LIKE '%ORSINI%'
                THEN 'SP'
                ELSE 'HCO'
            END AS channel,

            z.territory_id

        FROM com_edp_prd.com_intgr.distribution_sd_shipments s

        LEFT JOIN com_edp_prd.com_intgr.distribution_accounts a
            ON s.crx_account_id = a.crx_account_id

        LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping z
            ON LEFT(REGEXP_REPLACE(a.account_facility_zip, '[^0-9]', ''), 5)
             = LEFT(REGEXP_REPLACE(z.zipcode, '[^0-9]', ''), 5)

        WHERE s.ndc = '84976-0001-01'
          AND s.is_current = true
    )
    WHERE rn = 1
      AND territory_id IS NOT NULL
      AND channel = 'HCO'
),

q2 AS (

    SELECT DISTINCT
        s.crx_account_id

    FROM com_edp_prd.com_intgr.distribution_sd_shipments s

    LEFT JOIN com_edp_prd.com_intgr.distribution_accounts a
        ON s.crx_account_id = a.crx_account_id

    LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping z
        ON SUBSTRING(s.ship_to_address_postal_code, 1, 5) = z.zipcode

    WHERE s.ndc = '84976-0001-01'
      AND s.is_current = true

      AND CASE 
            WHEN UPPER(a.account_facility_name) LIKE '%ORSINI%'
            THEN 'SP'
            ELSE 'HCO'
          END = 'HCO'

      AND z.territory_id IS NOT NULL
)

SELECT 'ONLY_IN_Q2' AS source, *
FROM q2
WHERE crx_account_id NOT IN (
    SELECT crx_account_id FROM q1
)

UNION ALL

SELECT 'ONLY_IN_Q1' AS source, *
FROM q1
WHERE crx_account_id NOT IN (
    SELECT crx_account_id FROM q2
);